INSTALL PACKAGES and LOAD

In [3]:
# 1. Package Installation
cat("Installing packages...\n")
suppressWarnings(install.packages("sqldf", quiet = TRUE, repos = "http://cran.us.r-project.org"))
suppressWarnings(install.packages("ggplot2", quiet = TRUE, repos = "http://cran.us.r-project.org"))
cat("Packages installed successfully.\n\n")

# 2. Loading Libraries
cat("Loading libraries...\n")
suppressPackageStartupMessages(suppressWarnings(library(sqldf)))
suppressPackageStartupMessages(suppressWarnings(library(ggplot2)))
cat("Libraries loaded successfully.\n\n")

# 3. Loading Datasets
cat("Loading datasets...\n")
orders     <- read.csv("orders.csv")
deliveries <- read.csv("deliveries.csv")
customers  <- read.csv("customers.csv")
drivers    <- read.csv("drivers.csv")
complaints <- read.csv("complaints.csv")
incidents  <- read.csv("incidents.csv")
vehicles   <- read.csv("vehicles.csv")
hubs       <- read.csv("hubs.csv")
cat("Data loaded successfully.\n\n")

# 4. Displaying Summary Counts
cat("=== Dataset Row Counts ===\n")
cat("Orders:",     nrow(orders),     "\n")
cat("Deliveries:", nrow(deliveries), "\n")
cat("Customers:",  nrow(customers),  "\n")
cat("Complaints:", nrow(complaints), "\n")
cat("Incidents:",  nrow(incidents),  "\n")
cat("Vehicles:",   nrow(vehicles),   "\n")
cat("Hubs:",       nrow(hubs),       "\n")

Installing packages...
Packages installed successfully.

Loading libraries...
Libraries loaded successfully.

Loading datasets...
Data loaded successfully.

=== Dataset Row Counts ===
Orders: 1250 
Deliveries: 950 
Customers: 650 
Complaints: 320 
Incidents: 280 
Vehicles: 120 
Hubs: 8 


QUERY 1 (HUB PERFORMANCE)

In [ ]:
result1 <- sqldf("
  SELECT h.hub_id,
         h.hub_name,
         h.zone,
         h.hub_type,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END) AS ontime,
         SUM(CASE WHEN d.delivery_status = 'Late'   THEN 1 ELSE 0 END) AS late,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating,
         ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_id, h.hub_name, h.zone, h.hub_type
  ORDER BY failure_rate_pct DESC
")
print(result1)

  hub_id       hub_name      zone  hub_type total_deliveries failed ontime late
1    H08  Midtown Relay   Central  Charging              128     26     80    0
2    H05   Central Core   Central   Control              115     23     67    0
3    H06    Airport Hub   Airport  Dispatch              104     15     62    0
4    H04      West Gate      West  Dispatch              127     16     83    0
5    H01 North Exchange     North  Dispatch              136     17     93    0
6    H07  Riverside Hub Riverside Warehouse              115     14     76    0
7    H02     South Link     South  Dispatch              106     10     70    0
8    H03      East Dock      East Warehouse              119     11     85    0
  failure_rate_pct avg_customer_rating avg_cost
1            20.31                3.88    11.71
2            20.00                3.67    13.69
3            14.42                3.88    13.32
4            12.60                3.92    13.17
5            12.50                3.84  

SQL QUERY 2 (ZONE FAILURE ANALYSIS)

In [ ]:
result2 <- sqldf("
  SELECT o.pickup_zone,
         COUNT(*) AS total_orders,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
         SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END) AS ontime,
         SUM(CASE WHEN d.delivery_status = 'Late'   THEN 1 ELSE 0 END) AS late,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS failure_rate_pct,
         ROUND(AVG(o.order_value), 2) AS avg_order_value,
         ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_fuel_cost
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.pickup_zone
  ORDER BY failure_rate_pct DESC
")
print(result2)

   pickup_zone total_orders failed ontime late failure_rate_pct avg_order_value
1    RiverSide           66     14     40    0            21.21           86.23
2      Central           55     11     33    0            20.00           71.61
3      CENTRAL           55     11     28    0            20.00           97.49
4        North           37      7     24    0            18.92           86.58
5          Ctr           64     11     29    0            17.19           93.45
6        north           52      8     38    0            15.38           93.79
7        NORTH           46      7     30    0            15.22           89.13
8         EAST           78     11     53    0            14.10           93.85
9         West           51      7     36    0            13.73           86.45
10       South           83     10     58    0            12.05           90.65
11     Airport           67      8     41    0            11.94          103.75
12        WEST           63      7     4

 SQL QUERY 3 (DRIVER PERFORMANCE)

In [ ]:
result3 <- sqldf("
  SELECT dr.driver_id,
         dr.base_zone,
         dr.employment_type,
         dr.training_score,
         dr.driver_rating,
         COUNT(d.delivery_id) AS total_deliveries,
         SUM(d.manual_route_override_count) AS total_overrides,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating,
         ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_fuel_cost
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY dr.driver_id, dr.base_zone, dr.employment_type,
           dr.training_score, dr.driver_rating
  ORDER BY total_overrides DESC
  LIMIT 15
")
print(result3)

   driver_id base_zone employment_type training_score driver_rating
1       D127   CENTRAL        FullTime           61.5          4.19
2       D087     north        Contract           73.3          4.43
3       D130      WEST        FullTime           71.2          3.64
4       D108     South        FullTime           70.6          4.33
5       D131     SOUTH        FullTime           86.7          4.26
6       D069     NORTH        PartTime           61.5          5.00
7       D105 RiverSide        Contract           82.0          3.71
8       D017      EAST        PartTime             NA          4.34
9       D028     North        FullTime           83.0          4.07
10      D008     SOUTH        FullTime           84.1          3.88
11      D026     NORTH        PartTime           84.9          3.15
12      D104      WEST        FullTime           87.7          3.45
13      D027   AIRPORT        PartTime           74.3          3.70
14      D033     South        PartTime          

SQL Query 4 (COMPLAINTS vs DELIVERY OUTCOMES)

In [ ]:
complaints  <- read.csv("complaints.csv")
deliveries  <- read.csv("deliveries.csv")

result4 <- sqldf("
  SELECT c.complaint_type,
         c.severity,
         c.channel,
         d.delivery_status,
         COUNT(*) AS total_cases,
         ROUND(AVG(c.resolution_days), 2) AS avg_resolution_days,
         ROUND(AVG(c.compensation_amount), 2) AS avg_compensation
  FROM complaints c
  JOIN deliveries d ON c.order_id = d.order_id
  GROUP BY c.complaint_type, c.severity, c.channel, d.delivery_status
  ORDER BY total_cases DESC
")
print(result4)

       complaint_type severity channel delivery_status total_cases
1               Delay   Medium     App          OnTime           9
2               Delay   Medium   Phone          OnTime           8
3     DriverBehaviour   Medium   Phone          OnTime           8
4               Delay   Medium     App         Delayed           7
5        MissedPickup   Medium   Phone          OnTime           7
6               Delay   Medium Chatbot          OnTime           5
7   SupportExperience   Medium     App          OnTime           5
8            AppIssue   Medium     App          OnTime           4
9            AppIssue   Medium   Phone          OnTime           4
10              Delay      Low Chatbot          OnTime           4
11              Delay      Low   Email          OnTime           4
12    DriverBehaviour   Medium Chatbot          OnTime           4
13       MissedPickup   Medium     App          OnTime           4
14       MissedPickup   Medium Chatbot          OnTime        

SQL QUERY 5 (VEHICLE and INCIDENT ANALYSIS)

In [ ]:
result5 <- sqldf("
  SELECT v.vehicle_type,
         v.assigned_zone,
         v.maintenance_status,
         i.incident_type,
         i.severity AS incident_severity,
         COUNT(*) AS incident_count,
         ROUND(AVG(i.resolved_hours), 2) AS avg_resolved_hours,
         ROUND(AVG(v.battery_health_pct), 2) AS avg_battery_health,
         ROUND(AVG(v.odometer_km), 2) AS avg_odometer_km
  FROM incidents i
  JOIN deliveries d ON i.delivery_id = d.delivery_id
  JOIN vehicles v ON d.vehicle_id = v.vehicle_id
  GROUP BY v.vehicle_type, v.assigned_zone, v.maintenance_status,
           i.incident_type, i.severity
  ORDER BY incident_count DESC
")
print(result5)

    vehicle_type assigned_zone maintenance_status    incident_type
1       CargoVan          East             Active   RouteDeviation
2       CargoVan         NORTH             Active   CustomerNoShow
3       CargoVan         North             Active   RouteDeviation
4       CargoVan         SOUTH             Active     VehicleFault
5       CargoVan          West             Active     AppSyncError
6         Diesel       AIRPORT           InRepair     BatteryAlert
7         Diesel         North           InRepair     AppSyncError
8         Diesel         SOUTH           InRepair   SafetyNearMiss
9         Diesel         north           InRepair     AppSyncError
10            EV           Ctr             Active   RouteDeviation
11            EV           Ctr          Scheduled   RouteDeviation
12            EV         South          Scheduled TemperatureIssue
13            EV         South          Scheduled     VehicleFault
14            EV          WEST             Active     AppSyncE